# Test a saved Fraise model

This notebook loads a saved Fraise model from the project-local ./data/models directory and runs a small inference check.

It will:

- discover a completed model run and its archived model configuration;
- load the tokenizer from the explicitly supplied JSON path;
- score a short text with causal next-token loss;
- inspect the most likely next tokens;
- generate a short completion; and
- benchmark generation throughput in tokens per second.

Run the cells from top to bottom. If there are multiple model runs, set MODEL_RUN_NAME in the setup cell. Set TOKENIZER_PATH there to the tokenizer JSON file to use.

In [25]:
from __future__ import annotations

import hashlib
import json
import math
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "src" / "components").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the Fraise project so the project root "
        "can be found."
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import torch.nn.functional as F
from tokenizers import Tokenizer

from components.model import FraiseModel


MODEL_ROOT = PROJECT_ROOT / "data" / "models"
MODEL_RUN_NAME = None  # Set this when more than one run is available.
TOKENIZER_PATH = None  # Set the exact tokenizer JSON file to use.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Model root:   {MODEL_ROOT}")
print(f"Device:       {DEVICE}")

Project root: /home/munch/Projects/fraise
Model root:   /home/munch/Projects/fraise/data/models
Device:       cuda


In [26]:
def _read_json(path: Path) -> dict:
    with path.open(encoding="utf-8") as json_file:
        value = json.load(json_file)
    if not isinstance(value, dict):
        raise ValueError(f"Expected a JSON object in {path}")
    return value


def _resolve_model_path(
    run_dir: Path,
    model_root: Path,
    metadata: dict,
) -> Path | None:
    candidates: list[Path] = []
    final_model_path = metadata.get("final_model_path")

    if isinstance(final_model_path, str) and final_model_path:
        raw_path = Path(final_model_path).expanduser()
        if raw_path.is_absolute():
            candidates.append(raw_path)
        else:
            candidates.extend(
                [
                    Path.cwd() / raw_path,
                    PROJECT_ROOT / raw_path,
                    run_dir / raw_path,
                    model_root / raw_path,
                ]
            )

    candidates.extend(
        [
            run_dir / f"{run_dir.name}.pt",
            run_dir / "model.pt",
        ]
    )
    candidates.extend(
        path
        for path in sorted(run_dir.glob("*.pt"))
        if not path.name.startswith("checkpoint_step_")
    )
    candidates.extend(sorted(run_dir.glob("checkpoint_step_*.pt")))

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.is_file():
            return candidate.resolve()
    return None


def _discover_model_runs(model_root: Path) -> list[dict]:
    if not model_root.is_dir():
        return []

    records: list[dict] = []
    seen_model_paths: set[Path] = set()

    for metadata_path in sorted(model_root.rglob("model_meta.json")):
        try:
            metadata = _read_json(metadata_path)
        except (OSError, ValueError) as error:
            print(f"Skipping {metadata_path}: {error}")
            continue

        run_dir = metadata_path.parent
        model_path = _resolve_model_path(run_dir, model_root, metadata)
        if model_path is None or model_path in seen_model_paths:
            continue

        records.append(
            {
                "run_dir": run_dir,
                "metadata_path": metadata_path,
                "metadata": metadata,
                "model_path": model_path,
            }
        )
        seen_model_paths.add(model_path)

    # Keep the notebook useful for a model directory without model_meta.json.
    if not records:
        for model_path in sorted(model_root.rglob("*.pt")):
            if model_path.name.startswith("checkpoint_step_"):
                continue
            model_path = model_path.resolve()
            if model_path in seen_model_paths:
                continue
            records.append(
                {
                    "run_dir": model_path.parent,
                    "metadata_path": None,
                    "metadata": {},
                    "model_path": model_path,
                }
            )
            seen_model_paths.add(model_path)

    return records


MODEL_RUNS = _discover_model_runs(MODEL_ROOT)
if not MODEL_RUNS:
    raise FileNotFoundError(
        f"No model artifact was found under {MODEL_ROOT}. "
        "Expected a run directory containing a final .pt file."
    )

if MODEL_RUN_NAME is None:
    RUN = max(MODEL_RUNS, key=lambda record: record["model_path"].stat().st_mtime)
else:
    matches = [
        record
        for record in MODEL_RUNS
        if record["run_dir"].name == MODEL_RUN_NAME
        or str(record["run_dir"]) == str(Path(MODEL_RUN_NAME).expanduser())
    ]
    if len(matches) != 1:
        available = ", ".join(record["run_dir"].name for record in MODEL_RUNS)
        raise ValueError(
            f"Could not uniquely select MODEL_RUN_NAME={MODEL_RUN_NAME!r}. "
            f"Available runs: {available}"
        )
    RUN = matches[0]

RUN_DIR = RUN["run_dir"]
MODEL_PATH = RUN["model_path"]

print(f"Found {len(MODEL_RUNS)} model run(s):")
for record in MODEL_RUNS:
    marker = "*" if record is RUN else " "
    print(f" {marker} {record['run_dir'].name}: {record['model_path']}")


Found 3 model run(s):
   fraise_model: /home/munch/Projects/fraise/data/models/fraise_model/fraise_model.pt
   minsorhours: /home/munch/Projects/fraise/data/models/minsorhours/minsorhours.pt
 * unwantedchild: /home/munch/Projects/fraise/data/models/unwantedchild/unwantedchild.pt


In [27]:
metadata = RUN["metadata"]
manifest = metadata.get("manifest")
if not isinstance(manifest, dict):
    manifest = {}

required_model_keys = {
    "vocab_size",
    "residual_stream_dim",
    "head_dim",
    "qk_dim",
    "num_heads",
    "swiglu_up_dim",
    "num_layers",
    "max_seq_len",
}

model_config_value = manifest.get("model_config")
if isinstance(model_config_value, dict) and required_model_keys <= set(model_config_value):
    model_config = {
        key: int(model_config_value[key])
        for key in required_model_keys
    }
else:
    # Fall back to an archived model-config JSON when metadata is incomplete.
    config_candidates: list[Path] = []
    for config_name in metadata.get("config_files", []):
        if isinstance(config_name, str):
            config_candidates.append(RUN_DIR / config_name)
    config_candidates.extend(
        [
            RUN_DIR / "model_config.json",
            RUN_DIR / "mt.json",
        ]
    )

    model_config = None
    for config_path in config_candidates:
        if not config_path.is_file():
            continue
        try:
            candidate_config = _read_json(config_path)
        except (OSError, ValueError):
            continue
        if required_model_keys <= set(candidate_config):
            model_config = {
                key: int(candidate_config[key])
                for key in required_model_keys
            }
            break

    if model_config is None:
        raise ValueError(
            f"Could not find a complete model configuration for {MODEL_PATH}"
        )

tokenizer_manifest = manifest.get("tokenizer")
if not isinstance(tokenizer_manifest, dict):
    tokenizer_manifest = {}
expected_vocab_size = model_config["vocab_size"]
expected_fingerprint = tokenizer_manifest.get("fingerprint")


def _tokenizer_fingerprint(tokenizer: Tokenizer) -> str:
    return hashlib.sha256(tokenizer.to_str().encode("utf-8")).hexdigest()


if TOKENIZER_PATH is None:
    f = RUN["metadata"]["tokenizer_file"]
    if f:
        if path := Path(f).expanduser():
            if not path.is_absolute():
                path = RUN["run_dir"] / path
                TOKENIZER_PATH = path
            if path.is_file():
                TOKENIZER_PATH = path
    else:
        raise ValueError("Set TOKENIZER_PATH to the tokenizer JSON file to use.")

TOKENIZER_PATH = Path(TOKENIZER_PATH).expanduser()
if not TOKENIZER_PATH.is_absolute():
    TOKENIZER_PATH = PROJECT_ROOT / TOKENIZER_PATH
if not TOKENIZER_PATH.is_file():
    raise FileNotFoundError(f"Tokenizer file does not exist: {TOKENIZER_PATH}")

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
if tokenizer.get_vocab_size() != expected_vocab_size:
    raise ValueError(
        "Tokenizer/model vocabulary mismatch: "
        f"{tokenizer.get_vocab_size()} != {expected_vocab_size}"
    )
if expected_fingerprint and _tokenizer_fingerprint(tokenizer) != expected_fingerprint:
    raise ValueError("The supplied tokenizer does not match this model's recorded fingerprint.")


def _required_token_id(token: str, metadata_key: str) -> int:
    token_id = tokenizer.token_to_id(token)
    if token_id is None:
        token_id = tokenizer_manifest.get(metadata_key)
    if token_id is None:
        raise ValueError(f"Tokenizer is missing required token {token!r}")
    return int(token_id)


PAD_ID = _required_token_id("<|pad|>", "pad_token_id")
BOS_ID = _required_token_id("<|bos|>", "bos_token_id")
EOS_ID = _required_token_id("<|eos|>", "eos_token_id")

if tokenizer.get_vocab_size() != expected_vocab_size:
    raise ValueError(
        "Tokenizer/model vocabulary mismatch: "
        f"{tokenizer.get_vocab_size()} != {expected_vocab_size}"
    )

print(f"Selected run:  {RUN_DIR}")
print(f"Model config:  {model_config}")
print(f"Tokenizer:     {TOKENIZER_PATH}")
print(f"Special IDs:   pad={PAD_ID}, bos={BOS_ID}, eos={EOS_ID}")

Selected run:  /home/munch/Projects/fraise/data/models/unwantedchild
Model config:  {'qk_dim': 32, 'residual_stream_dim': 256, 'num_layers': 32, 'max_seq_len': 512, 'vocab_size': 8192, 'swiglu_up_dim': 1024, 'num_heads': 12, 'head_dim': 64}
Tokenizer:     /home/munch/Projects/fraise/data/models/unwantedchild/tokenizer.json
Special IDs:   pad=0, bos=1, eos=2


In [ ]:
# The final artifact is a state_dict; this also accepts a training checkpoint
# containing a model_state_dict entry.
loaded = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=False,  # The artifact is a local, trusted Fraise run.
)
if isinstance(loaded, dict) and "model_state_dict" in loaded:
    state_dict = loaded["model_state_dict"]
else:
    state_dict = loaded

if not isinstance(state_dict, dict):
    raise TypeError(f"Expected a state dict in {MODEL_PATH}, got {type(state_dict)}")

model = FraiseModel(model_config)
model.load_state_dict(state_dict, strict=True)
model.to(DEVICE)
model = torch.compile(model, fullgraph=True)
model.eval()

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Loaded {parameter_count:,} parameters from {MODEL_PATH.name}")
print(f"Model context length: {model.max_seq_len} tokens")

RuntimeError: Error(s) in loading state_dict for OptimizedModule:
	Missing key(s) in state_dict: "_orig_mod.token_embedding.embedding.weight", "_orig_mod.transformer.layers.0.mha.qkv_projection.weight", "_orig_mod.transformer.layers.0.mha.output_projection.weight", "_orig_mod.transformer.layers.0.swiGLU.value.weight", "_orig_mod.transformer.layers.0.swiGLU.value.bias", "_orig_mod.transformer.layers.0.swiGLU.gate.weight", "_orig_mod.transformer.layers.0.swiGLU.gate.bias", "_orig_mod.transformer.layers.0.swiGLU.output.weight", "_orig_mod.transformer.layers.0.swiGLU.output.bias", "_orig_mod.transformer.layers.0.layernorm1.weight", "_orig_mod.transformer.layers.0.layernorm1.bias", "_orig_mod.transformer.layers.0.layernorm2.weight", "_orig_mod.transformer.layers.0.layernorm2.bias", "_orig_mod.transformer.layers.1.mha.qkv_projection.weight", "_orig_mod.transformer.layers.1.mha.output_projection.weight", "_orig_mod.transformer.layers.1.swiGLU.value.weight", "_orig_mod.transformer.layers.1.swi

In [ ]:
def _encode_body(text: str) -> list[int]:
    return tokenizer.encode(text, add_special_tokens=False).ids


def _prompt_ids(text: str) -> list[int]:
    return [BOS_ID, *_encode_body(text)]


def _evaluation_ids(text: str) -> list[int]:
    return [BOS_ID, *_encode_body(text), EOS_ID]


def _context_window(token_ids: list[int]) -> list[int]:
    return token_ids[-model.max_seq_len:]


def _logits_for_context(token_ids: list[int]) -> torch.Tensor:
    context = _context_window(token_ids)
    input_ids = torch.tensor(
        [context],
        dtype=torch.long,
        device=DEVICE,
    )
    with torch.inference_mode():
        return model(input_ids)[0, -1]


def score_text(text: str) -> dict[str, float | int]:
    token_ids = _evaluation_ids(text)
    # The model accepts max_seq_len input tokens and predicts the following token.
    token_ids = token_ids[: model.max_seq_len + 1]
    input_ids = torch.tensor(
        [token_ids[:-1]],
        dtype=torch.long,
        device=DEVICE,
    )
    targets = torch.tensor(
        [token_ids[1:]],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.inference_mode():
        logits = model(input_ids)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
        )

    loss_value = float(loss.cpu())
    return {
        "tokens_scored": len(token_ids) - 1,
        "loss": loss_value,
        "perplexity": math.exp(min(loss_value, 80.0)),
    }


def decode_token(token_id: int) -> str:
    decoded = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    raw = tokenizer.id_to_token(int(token_id))
    return f"{raw!r} -> {decoded!r}"

In [ ]:
TEST_TEXT = (
    "Fraise explores efficient language modeling with compressed "
    "representations."
)
score = score_text(TEST_TEXT)

print(f"Scored tokens: {score['tokens_scored']}")
print(f"Cross-entropy: {score['loss']:.4f}")
print(f"Perplexity:    {score['perplexity']:.2f}")

assert math.isfinite(score["loss"])
assert score["tokens_scored"] > 0

Scored tokens: 16
Cross-entropy: 5.1462
Perplexity:    171.77


In [ ]:
def show_top_next_tokens(text: str, top_k: int = 10) -> None:
    logits = _logits_for_context(_prompt_ids(text))
    probabilities = torch.softmax(logits, dim=-1)
    values, token_ids = torch.topk(
        probabilities,
        k=min(top_k, probabilities.numel()),
    )

    print(f"Prompt: {text!r}")
    for rank, (probability, token_id) in enumerate(
        zip(values.cpu().tolist(), token_ids.cpu().tolist()),
        start=1,
    ):
        print(
            f"{rank:>2}. id={token_id:>5} "
            f"p={probability:.4f} "
            f"{decode_token(token_id)}"
        )


show_top_next_tokens("The future of small language models is", top_k=10)

Prompt: 'The future of small language models is'
 1. id=  266 p=0.1178 'Ġthe' -> ' the'
 2. id=  262 p=0.0487 'Ġa' -> ' a'
 3. id=  350 p=0.0308 'Ġthat' -> ' that'
 4. id=  289 p=0.0284 'Ġto' -> ' to'
 5. id= 1504 p=0.0204 'Ġknown' -> ' known'
 6. id=  292 p=0.0171 'Ġin' -> ' in'
 7. id=  505 p=0.0167 'Ġnot' -> ' not'
 8. id= 1729 p=0.0146 'Ġbased' -> ' based'
 9. id= 1048 p=0.0137 'Ġbeing' -> ' being'
10. id=  352 p=0.0077 'Ġan' -> ' an'


In [ ]:
def generate(
    prompt: str,
    *,
    max_new_tokens: int = 48,
    temperature: float = 0.0,
    top_k: int | None = 50,
    top_p: float | None = 0.95,
) -> tuple[str, list[int]]:
    token_ids = _prompt_ids(prompt)

    for _ in range(max_new_tokens):
        logits = _logits_for_context(token_ids)

        if temperature <= 0:
            next_token_id = int(torch.argmax(logits).item())
        else:
            if top_k is not None and top_p is not None:
                raise ValueError("Cannot use both top_k and top_p sampling together.")
            logits = logits / temperature

            # Top-k filtering
            if top_k is not None:
                k = min(top_k, logits.numel())
                threshold = torch.topk(logits, k).values[-1]
                logits = logits.masked_fill(logits < threshold, float("-inf"))

            # Top-p (nucleus) filtering
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(
                    logits, descending=True
                )
                sorted_probs = torch.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                # Remove tokens after cumulative probability exceeds top_p.
                # Shift right so we always keep at least the highest-prob token.
                remove = cumulative_probs > top_p
                remove[1:] = remove[:-1].clone()
                remove[0] = False

                sorted_logits[remove] = float("-inf")

                # Scatter back into original vocabulary order
                filtered_logits = torch.full_like(logits, float("-inf"))
                filtered_logits.scatter_(0, sorted_indices, sorted_logits)
                logits = filtered_logits

            probabilities = torch.softmax(logits, dim=-1)
            next_token_id = int(
                torch.multinomial(probabilities, num_samples=1).item()
            )

        token_ids.append(next_token_id)

        if next_token_id == EOS_ID:
            break

    return (
        tokenizer.decode(token_ids[1:], skip_special_tokens=True),
        token_ids,
    )

PROMPT = "The capital of france is"
generated_text, generated_ids = generate(
    PROMPT,
    max_new_tokens=300,
    temperature=0.0,
    top_k=50,
    top_p=None,
)
print(f"Prompt:     {PROMPT}")
print(f"Generated:  {generated_text}")
print(f"New tokens: {len(generated_ids) - len(_prompt_ids(PROMPT))}")

Prompt:     The capital of france is
Generated:  The capital of france is the most important part of the country. The country is located in the province of Lake Sinai, the capital of the province of Sinai, and the province of Sinai, the province of Sinai, the province of Sinai, and the province of Sinai, the province of Sinai, and the province of Sinai, the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the province of Sinai, and the Sicily, and the Sicily, and the Sicily, and the Sicily, and the Sicily, and the Sicily, and the Sicily, and the Sicily, and the Sicily, and the Sicily, and the Sicily, and the Sicily,
New tokens: 300


## Generation throughput

This measures single prompt, greedy decode throughput in generated tokens per second (TPS). Each token uses the same full context model pass as `generate`, without a KV cache. Warmup runs are excluded, the GPU is synchronized around each timed run, and generation continues for a fixed number of tokens even if EOS appears. Prompt tokenization and final text decoding are excluded from the timer.

In [ ]:
import time


def benchmark_generation_tps(
    prompt: str,
    *,
    max_new_tokens: int = 32,
    warmup_runs: int = 1,
    timed_runs: int = 3,
) -> dict[str, float | int]:
    if max_new_tokens <= 0 or warmup_runs < 0 or timed_runs <= 0:
        raise ValueError("Use positive max_new_tokens and timed_runs, and nonnegative warmup_runs.")

    prompt_ids = _prompt_ids(prompt)
    elapsed_seconds = []

    for run in range(warmup_runs + timed_runs):
        token_ids = prompt_ids.copy()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()

        for _ in range(max_new_tokens):
            logits = _logits_for_context(token_ids)
            token_ids.append(int(torch.argmax(logits).item()))

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        if run >= warmup_runs:
            elapsed_seconds.append(elapsed)

    total_tokens = max_new_tokens * timed_runs
    total_seconds = sum(elapsed_seconds)
    return {
        "prompt_tokens": len(prompt_ids),
        "tokens_per_run": max_new_tokens,
        "timed_runs": timed_runs,
        "total_seconds": total_seconds,
        "tokens_per_second": total_tokens / total_seconds,
    }


BENCHMARK_PROMPT = "The future of small language models is"
BENCHMARK_TOKENS = 32
BENCHMARK_WARMUP_RUNS = 1
BENCHMARK_TIMED_RUNS = 3

benchmark = benchmark_generation_tps(
    BENCHMARK_PROMPT,
    max_new_tokens=BENCHMARK_TOKENS,
    warmup_runs=BENCHMARK_WARMUP_RUNS,
    timed_runs=BENCHMARK_TIMED_RUNS,
)
print(f"Device:           {DEVICE}")
print(f"Prompt tokens:    {benchmark['prompt_tokens']}")
print(f"Generated tokens: {benchmark['tokens_per_run']} per run")
print(f"Timed runs:       {benchmark['timed_runs']}")
print(f"Total time:       {benchmark['total_seconds']:.3f} s")
print(f"Throughput:       {benchmark['tokens_per_second']:.2f} tokens/s")

Device:           cuda
Prompt tokens:    8
Generated tokens: 32 per run
Timed runs:       3
Total time:       0.969 s
Throughput:       99.08 tokens/s
